##### Copyright 2026 Google LLC.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: GitHub Issue Analyzer

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/GitHub_issue_analyzer.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>

<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/ROHANDWIVEDI2005" target="_blank" title="View Rohan's profile on GitHub">
        <img src="https://avatars.githubusercontent.com/u/165134541?v=4?size=100"
             alt="Rohan Dwivedi's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/rohandwivedi2005" target="_blank"><font color='#217bfe'><strong>Rohan Dwivedi</strong></font></a>.</font></h2>
      <h5><font color='black'><a href="www.linkedin.com/in/rohan-dwivedi-a67664296"><font color="#078efb">Find me at linked-in</font></a> - See <a href="https://github.com/rohandwivedi2005" target="_blank"><font color="#078efb"><strong>Rohan</strong></font></a> other notebooks <a href="https://github.com/search?q=repo%3Agoogle-gemini%2Fcookbook%20%rohandwivedi2005%22&type=code" target="_blank"><font color="#078efb">here</font></a>.</h5></font><br>
      <!-- Footer -->
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>

## Introduction
- This notebook demonstrates how to use the Gemini API with function calling to interact with the GitHub API and extract and analyze information about issues in a repository.
- To achieve this goal, you will be using the `PyGithub` python library to extract information from GitHub which will later be analyzed by Gemini to present the answers to your queries.

## Why PyGithub?
PyGitHub is a Python library to access the GitHub REST API. This library enables you to manage GitHub resources such as repositories, user profiles, and organizations in your Python applications.

## Setup

### Install SDK

In [2]:
%pip install -U -q "google-genai>=1.0.0"  # Install the Python SDK

# Always set at least 1.0.0 as the minimal version as there were breaking
# changes through the previous versions
# Of course, if your notebook uses a new feature and needs a more recent
# version, set the right minimum version to indicate when the feature was
# introduced.
# Always test your notebook with that fixed version (eg. '==1.0.0') to make.
# sure it's really the minimum version.


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google.colab import userdata
from google import genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](./Get_started_thinking.ipynb) for more details and in particular learn how to switch the thinking off).

In [4]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

# Ideally order the model by "cabability" ie. generation then within generation
# 8b/flash-lite then flash then pro

## Install dependencies

In [ ]:
%pip install -U -q "PyGithub"  # Install the PyGithub library

You will need a GitHub Personal Access Token stored as a Colab Secret named **GITHUB_API_KEY**.

1. Navigate to your GitHub account settings, select `Developer settings`
2. Then choose Personal access tokens and click on Tokens (classic)
3. From there, you can generate a new token with the specific scopes (permissions) required for your application

In [ ]:
GITHUB_API_KEY = userdata.get('GITHUB_API_KEY')

## Define GitHub API functions as tools
Now, define the Python functions that will be used as tools by the Gemini model. These functions will interact with the GitHub API using the PyGithub library. The docstrings are crucial as they provide the model with the necessary information to understand what each function does and what arguments it expects.

In [ ]:
from github import Github
from datetime import datetime, timezone
from collections import Counter

g = Github(GITHUB_API_KEY)

def get_github_issues(repository_name: str):
    """Get all open issues."""
    try:
        repo = g.get_repo(repository_name)
        issues = repo.get_issues(state='open')
        return [{
            'title': issue.title,
            'number': issue.number,
            'url': issue.html_url
        } for issue in issues]
    except Exception as e:
        print(f"An error occurred: {e}")
        return []




def get_issues_by_label(repository_name: str, label: str):
    """Get all open issues filtered by a specific label (e.g. 'bug', 'enhancement')."""
    try:
        repo = g.get_repo(repository_name)
        issues = repo.get_issues(state='open', labels=[label])
        return [{
            'title': issue.title,
            'number': issue.number,
            'url': issue.html_url,
            'created_at': issue.created_at.strftime('%Y-%m-%d'),
            'author': issue.user.login
        } for issue in issues if not issue.pull_request]
    except Exception as e:
        print(f"An error occurred: {e}")
        return []


def get_stale_issues(repository_name: str, days_inactive: int = 30):
    """Get open issues with no activity for N+ days."""
    try:
        repo = g.get_repo(repository_name)
        issues = repo.get_issues(state='open')
        stale = []

        for issue in issues:
            if issue.pull_request:
                continue
            last_activity = issue.updated_at
            inactive_days = (datetime.now(timezone.utc) - last_activity).days

            if inactive_days >= days_inactive:
                stale.append({
                    'title': issue.title,
                    'number': issue.number,
                    'url': issue.html_url,
                    'inactive_days': inactive_days,
                    'last_updated': last_activity.strftime('%Y-%m-%d')
                })

        return sorted(stale, key=lambda x: x['inactive_days'], reverse=True)

    except Exception as e:
        print(f"An error occurred: {e}")
        return []


def get_unassigned_issues(repository_name: str):
    """Get all open issues that have no assignee."""
    try:
        repo = g.get_repo(repository_name)
        issues = repo.get_issues(state='open')
        return [{
            'title': issue.title,
            'number': issue.number,
            'url': issue.html_url,
            'labels': [l.name for l in issue.labels],
            'created_at': issue.created_at.strftime('%Y-%m-%d')
        } for issue in issues if not issue.pull_request and not issue.assignee]
    except Exception as e:
        print(f"An error occurred: {e}")
        return []


def get_issue_summary(repository_name: str):
    """Get a high-level summary: total counts, top labels, most active contributors."""
    try:
        repo = g.get_repo(repository_name)
        issues = repo.get_issues(state='open')
        label_counter = Counter()
        author_counter = Counter()
        total = 0

        for issue in issues:
            if issue.pull_request:
                continue
            total += 1
            author_counter[issue.user.login] += 1
            for label in issue.labels:
                label_counter[label.name] += 1

        return {
            'total_open_issues': total,
            'top_labels': label_counter.most_common(5),
            'top_contributors': author_counter.most_common(5)
        }

    except Exception as e:
        print(f"An error occurred: {e}")
        return {}


def search_issues(repository_name: str, keyword: str):
    """Search open issues whose title or body contains a keyword."""
    try:
        repo = g.get_repo(repository_name)
        issues = repo.get_issues(state='open')
        keyword_lower = keyword.lower()
        return [{
            'title': issue.title,
            'number': issue.number,
            'url': issue.html_url,
            'matched_in': 'title' if keyword_lower in issue.title.lower() else 'body'
        } for issue in issues
          if not issue.pull_request
          and (keyword_lower in issue.title.lower()
               or (issue.body and keyword_lower in issue.body.lower()))]
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

## Interact with the model

Now it's time to interact with the Gemini model. Start a ChatSession and pass the functions as a tools. The model will automatically call the function with the correct arguments based on the prompt.

In [13]:
client = genai.Client(api_key=GOOGLE_API_KEY)
chat = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": [
            get_github_issues,
            get_issues_by_label,
            get_stale_issues,
            get_unassigned_issues,
            get_issue_summary,
            search_issues
        ]
    }
)

# ── Now just ask naturally — model picks the right tool automatically ─────────
questions = [
    "Show me the open issues in google-gemini/cookbook.",
    "Find any stale issues inactive for 60+ days in google-gemini/cookbook.",
    "Show me all unassigned issues in google-gemini/cookbook.",
    "Search for issues mentioning 'authentication' in google-gemini/cookbook.",
    "Give me a summary of issues in google-gemini/cookbook.",
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    response = chat.send_message(question)
    print(response.text)



Q: Show me the open issues in google-gemini/cookbook.
There are many open issues and pull requests in the [google-gemini/cookbook](https://github.com/google-gemini/cookbook/issues) repository. Here are some of the most recent ones:

### **Open Issues (Recent)**
*   **#1149:** [Linking GCP billing account does not upgrade Gemini API from free-tier quotas](https://github.com/google-gemini/cookbook/issues/1149)
*   **#1144:** [Monthly issue metrics report](https://github.com/google-gemini/cookbook/issues/1144)
*   **#1125:** [Add Example Notebook: Context-Aware Support Ticket Assistant](https://github.com/google-gemini/cookbook/issues/1125)
*   **#1123:** [Live API for just quick transcription? Or is gemini-2.0-flash-lite fastest possible?](https://github.com/google-gemini/cookbook/issues/1123)
*   **#1122:** [API_KEY login not supported for genai.client](https://github.com/google-gemini/cookbook/issues/1122)
*   **#1108:** [Live API does not perceive visual information and hallucinates]

## Next Steps
### Useful API references:
- [Pygithub](https://pygithub.readthedocs.io/en/latest/apis.html)
- [GeminiAPI](https://ai.google.dev/gemini-api/docs)

### Continue your discovery of the Gemini API

 - [Apollo_11](https://github.com/google-gemini/cookbook/blob/main/examples/Apollo_11.ipynb)
 - [Guess_the_shape](https://github.com/google-gemini/cookbook/blob/main/examples/Guess_the_shape.ipynb)
